# Bolsa Família 2020 — Análise Exploratória

**Fonte:** Portal da Transparência — Pagamentos Bolsa Família 2020  
**Volume:** 12 CSVs (~17.63 GB), ~160 milhões de linhas  
**Engine:** DuckDB (lê CSV direto, sem carregar em memória)

**Pipeline:** Bronze (CSVs brutos) → Silver (view limpa e tipada) → Gold (resultados exportados)

---

## Setup

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
from pathlib import Path

plt.rcParams['figure.figsize'] = (13, 5)
sns.set_theme(style='whitegrid')

# Configurações de caminho
ROOT   = Path(os.path.abspath('')).parent
BRONZE = str(ROOT / 'data' / 'bronze' / '*.csv')
GOLD   = str(ROOT / 'data' / 'gold')
os.makedirs(GOLD, exist_ok=True)

con = duckdb.connect()

# Lê as colunas do arquivo CSV para criar as variáveis de coluna
_cols = con.execute(f"""
    SELECT * FROM read_csv('{BRONZE}', delim=';', header=true, encoding='latin-1', ignore_errors=true) LIMIT 0
""").df().columns.tolist()

MES_COMP = f'"{_cols[0]}"'  # MÊS COMPETÊNCIA
MES_REF  = f'"{_cols[1]}"'  # MÊS REFERÊNCIA
UF       = f'"{_cols[2]}"'  # UF
COD_MUN  = f'"{_cols[3]}"'  # CÓDIGO MUNICÍPIO SIAFI
NOME_MUN = f'"{_cols[4]}"'  # NOME MUNICÍPIO
NIS      = f'"{_cols[6]}"'  # NIS FAVORECIDO
NOME_FAV = f'"{_cols[7]}"'  # NOME FAVORECIDO
VALOR    = f'"{_cols[8]}"'  # VALOR PARCELA

print('DuckDB', duckdb.__version__)
print('Colunas:', _cols)

## Camada Silver — view limpa e tipada

Centraliza toda a limpeza dos dados brutos:
- Nomes de colunas normalizados (snake_case)
- `valor_parcela` convertido de string BR para DOUBLE
- Coluna `regiao` derivada da UF
- Encoding e separadores tratados uma única vez

Todas as queries de análise usam `silver` diretamente.

In [ ]:
con.execute(f"""
CREATE OR REPLACE VIEW silver AS
SELECT
    {MES_COMP}       AS mes_competencia,
    {MES_REF}        AS mes_referencia,
    TRIM({UF})       AS uf,
    {COD_MUN}        AS cod_municipio,
    TRIM({NOME_MUN}) AS municipio,
    {NIS}            AS nis,
    {NOME_FAV}       AS nome_favorecido,
    TRY_CAST(
        REPLACE(REPLACE({VALOR}, '.', ''), ',', '.') AS DOUBLE
    )                AS valor_parcela,
    CASE TRIM({UF})
        WHEN 'AM' THEN 'Norte'        WHEN 'PA' THEN 'Norte'
        WHEN 'AC' THEN 'Norte'        WHEN 'RO' THEN 'Norte'
        WHEN 'RR' THEN 'Norte'        WHEN 'AP' THEN 'Norte'
        WHEN 'TO' THEN 'Norte'
        WHEN 'MA' THEN 'Nordeste'     WHEN 'PI' THEN 'Nordeste'
        WHEN 'CE' THEN 'Nordeste'     WHEN 'RN' THEN 'Nordeste'
        WHEN 'PB' THEN 'Nordeste'     WHEN 'PE' THEN 'Nordeste'
        WHEN 'AL' THEN 'Nordeste'     WHEN 'SE' THEN 'Nordeste'
        WHEN 'BA' THEN 'Nordeste'
        WHEN 'MT' THEN 'Centro-Oeste' WHEN 'MS' THEN 'Centro-Oeste'
        WHEN 'GO' THEN 'Centro-Oeste' WHEN 'DF' THEN 'Centro-Oeste'
        WHEN 'MG' THEN 'Sudeste'      WHEN 'ES' THEN 'Sudeste'
        WHEN 'RJ' THEN 'Sudeste'      WHEN 'SP' THEN 'Sudeste'
        WHEN 'PR' THEN 'Sul'          WHEN 'SC' THEN 'Sul'
        WHEN 'RS' THEN 'Sul'
        ELSE 'Outro'
    END              AS regiao
FROM read_csv(
    '{BRONZE}',
    delim         = ';',
    header        = true,
    encoding      = 'latin-1',
    ignore_errors = true
)
WHERE {VALOR} IS NOT NULL
  AND TRIM({UF}) != ''
""")
print('View silver criada.')

In [ ]:
# Exibe as primeiras linhas da view silver
con.execute('DESCRIBE silver').df()

In [ ]:
#Amostra dos dados
con.execute('SELECT * FROM silver LIMIT 5').df()

---
## Análise Exploratória — 15 Perguntas
---

## Principais Achados

Antes de entrar nas perguntas uma a uma, cinco coisas que os dados deixam claro:

- O programa cresceu 7,2% em abril de 2020 e nunca voltou ao patamar anterior — o lockdown está nos dados.
- O Nordeste domina em número de famílias (49,4%), mas o Norte tem a maior parcela média por família (R$ 210).
- Os top 5 estados concentram 46% do gasto — mais distribuído do que parece à primeira vista.
- Municípios indígenas da Amazônia têm as maiores parcelas médias: Uiramutã (RR) chega a R$ 445 por família.
- 86,8% dos beneficiários receberam o programa todos os 12 meses de 2020.

---

### P1. Qual o volume total de beneficiários e pagamentos em 2020?

In [ ]:
con.execute("""
    SELECT
        COUNT(*)                        AS total_pagamentos,
        COUNT(DISTINCT nis)             AS beneficiarios_unicos,
        ROUND(SUM(valor_parcela), 2)    AS total_pago_reais,
        ROUND(AVG(valor_parcela), 2)    AS media_parcela
    FROM silver
""").df()

168 milhões de pagamentos para 14,7 milhões de pessoas — na prática, 1 em cada 14 brasileiros recebeu o programa em algum momento de 2020. No total, R$ 32 bilhões saíram do governo federal só por esse canal, com parcela média de R$ 190. Para ter noção do tamanho: são R$ 87 milhões por dia, incluindo fins de semana e os meses de lockdown.

### P2. Como o número de beneficiários variou mês a mês?

In [ ]:
df_p2 = con.execute("""
    SELECT
        mes_competencia     AS mes,
        COUNT(*)            AS pagamentos,
        COUNT(DISTINCT nis) AS beneficiarios_unicos
    FROM silver
    GROUP BY mes ORDER BY mes
""").df()

fig, ax = plt.subplots()
ax.plot(df_p2['mes'].astype(str), df_p2['beneficiarios_unicos'] / 1e6, marker='o', color='steelblue')
ax.set_title('Beneficiários únicos por mês')
ax.set_ylabel('Milhões')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(f'{GOLD}/p2_beneficiarios_por_mes.png', dpi=150)
plt.show()
df_p2

De janeiro a março, o programa atendia em torno de 13,2 milhões de famílias por mês. Em abril o número salta para 14,3 milhões — e fica assim até dezembro. O salto coincide exatamente com o início do lockdown nacional. Não voltou ao patamar anterior em nenhum mês do ano.

### P3. Como o total pago (R$) variou mês a mês?

In [ ]:
df_p3 = con.execute("""
    SELECT
        mes_competencia      AS mes,
        SUM(valor_parcela)   AS total_pago
    FROM silver
    GROUP BY mes ORDER BY mes
""").df()

fig, ax = plt.subplots()
ax.bar(df_p3['mes'].astype(str), df_p3['total_pago'] / 1e9, color='seagreen')
ax.set_title('Total pago por mês (R$ bilhões)')
ax.set_ylabel('R$ bilhões')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(f'{GOLD}/p3_total_pago_por_mes.png', dpi=150)
plt.show()
df_p3

O salto de R$ 2,5 bi para R$ 2,7 bi em abril (+7,2%) fica bem visível no gráfico. O que chama atenção não é a magnitude do aumento, mas o fato de que ele permanece: de julho a dezembro o programa opera em torno de R$ 2,73 bi por mês, sem queda. Não foi antecipação de parcelas — foram famílias que entraram no programa e ficaram.

### P4. Qual a variação percentual mensal no total pago?

In [ ]:
df_p4 = con.execute("""
    WITH mensal AS (
        SELECT mes_competencia AS mes, SUM(valor_parcela) AS total
        FROM silver GROUP BY mes ORDER BY mes
    )
    SELECT
        mes,
        ROUND(total, 2) AS total_pago,
        ROUND(
            100.0 * (total - LAG(total) OVER (ORDER BY mes))
                  / LAG(total) OVER (ORDER BY mes),
        2) AS variacao_pct
    FROM mensal
""").df()

cores = ['seagreen' if v >= 0 else 'tomato' for v in df_p4['variacao_pct'].fillna(0)]
fig, ax = plt.subplots()
ax.bar(df_p4['mes'].astype(str), df_p4['variacao_pct'].fillna(0), color=cores)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Variação % mensal no total pago')
ax.set_ylabel('%')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(f'{GOLD}/p4_variacao_mensal.png', dpi=150)
plt.show()
df_p4

+7,2% em abril, depois nada expressivo até dezembro. Esse padrão indica que o governo fez um ajuste único e deixou o programa rodar estável a partir daí. Se tivesse sido uma antecipação de parcelas, o mês seguinte mostraria queda — não mostra.

### P5. Quais estados concentram mais beneficiários únicos?

In [ ]:
df_p5 = con.execute("""
    SELECT
        uf,
        COUNT(DISTINCT nis)          AS beneficiarios_unicos,
        ROUND(SUM(valor_parcela), 2) AS total_pago
    FROM silver
    GROUP BY uf
    ORDER BY beneficiarios_unicos DESC
""").df()

fig, ax = plt.subplots(figsize=(14, 5))
sns.barplot(data=df_p5, x='uf', y='beneficiarios_unicos', ax=ax, palette='Blues_r')
ax.set_title('Beneficiários únicos por estado')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.tight_layout()
plt.savefig(f'{GOLD}/p5_beneficiarios_por_uf.png', dpi=150)
plt.show()
df_p5

Bahia lidera com 1,89 milhão de beneficiários únicos. São Paulo aparece em segundo com 1,69 milhão, o que parece estranho para o estado mais rico do país — até você lembrar que SP tem 22% da população brasileira. Mesmo com menor taxa de pobreza, o volume absoluto é alto. P7 coloca isso em proporção.

### P6. Qual região concentra mais beneficiários e maior volume financeiro?

In [ ]:
df_p6 = con.execute("""
    SELECT
        regiao,
        COUNT(DISTINCT nis)                AS beneficiarios_unicos,
        ROUND(SUM(valor_parcela) / 1e9, 2) AS total_pago_bilhoes,
        ROUND(AVG(valor_parcela), 2)       AS media_parcela
    FROM silver
    GROUP BY regiao
    ORDER BY beneficiarios_unicos DESC
""").df()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(df_p6['beneficiarios_unicos'], labels=df_p6['regiao'], autopct='%1.1f%%', startangle=140)
axes[0].set_title('Beneficiários únicos por região')
axes[1].pie(df_p6['total_pago_bilhoes'], labels=df_p6['regiao'], autopct='%1.1f%%', startangle=140)
axes[1].set_title('Total pago por região (R$ bi)')
plt.tight_layout()
plt.savefig(f'{GOLD}/p6_regioes.png', dpi=150)
plt.show()
df_p6

O Nordeste concentra quase metade dos beneficiários (49,4%) e pouco mais da metade do gasto total (51,1%). Mas o dado mais interessante aparece no Norte: com apenas 12,5% dos beneficiários, a região tem a maior parcela média do país — R$ 210 por família, contra R$ 190 de média nacional. Menos famílias, mas cada uma recebendo mais.

### P7. Qual o percentual de beneficiários de cada estado em relação ao total nacional?

In [ ]:
con.execute("""
    WITH total AS (
        SELECT COUNT(DISTINCT nis) AS n FROM silver
    )
    SELECT
        uf,
        COUNT(DISTINCT nis) AS beneficiarios,
        ROUND(100.0 * COUNT(DISTINCT nis) / MAX(total.n), 2) AS pct_nacional
    FROM silver, total
    GROUP BY uf
    ORDER BY pct_nacional DESC
""").df()

BA com 12,82%, SP com 11,47%. Mas repara: SP tem ~22% da população nacional e apenas 11,47% dos beneficiários do programa. Isso quer dizer que SP está subrepresentado em relação ao seu tamanho — o que faz sentido, dado que é o estado com maior renda per capita do país.

### P8. Qual estado tem o maior valor médio de parcela?

In [ ]:
df_p8 = con.execute("""
    SELECT
        uf,
        ROUND(AVG(valor_parcela), 2)    AS media_parcela,
        ROUND(MEDIAN(valor_parcela), 2) AS mediana_parcela,
        COUNT(*)                        AS pagamentos
    FROM silver
    GROUP BY uf
    ORDER BY media_parcela DESC
""").df()

fig, ax = plt.subplots(figsize=(14, 5))
sns.barplot(data=df_p8, x='uf', y='media_parcela', ax=ax, palette='Oranges_r')
ax.set_title('Valor médio da parcela por estado (R$)')
ax.set_ylabel('R$')
plt.tight_layout()
plt.savefig(f'{GOLD}/p8_media_parcela_por_uf.png', dpi=150)
plt.show()
df_p8.head(10)

Acre com média de R$ 272, Rondônia com R$ 155 — uma diferença de R$ 117 entre dois estados vizinhos da mesma região. Não é arbitrariedade do programa: o Bolsa Família tem um componente variável que aumenta conforme o número de filhos e a profundidade da pobreza. No Acre, as famílias têm em média mais crianças em situação de vulnerabilidade do que em Rondônia — e o programa paga de acordo.

### P9. Como se distribui o valor das parcelas? (histograma)

In [ ]:
df_p9 = con.execute("""
    SELECT valor_parcela AS valor
    FROM silver
    USING SAMPLE 500000
""").df()

fig, ax = plt.subplots()
ax.hist(df_p9['valor'].dropna(), bins=60, color='steelblue', edgecolor='white')
ax.set_title('Distribuição dos valores de parcela — amostra 500k')
ax.set_xlabel('Valor (R$)')
ax.set_ylabel('Frequência')
plt.tight_layout()
plt.savefig(f'{GOLD}/p9_distribuicao_valores.png', dpi=150)
plt.show()
df_p9['valor'].describe().round(2)

O histograma não é aleatório. Os picos em valores específicos mostram que existe uma estrutura por trás: cada pico corresponde a uma faixa do programa (valor base + componentes por número de filhos). Dá pra "ver" as regras do Bolsa Família só olhando para a distribuição dos valores.

### P10. Top 20 municípios por total pago

In [ ]:
df_p10 = con.execute("""
    SELECT
        municipio,
        uf,
        COUNT(*)                         AS pagamentos,
        ROUND(SUM(valor_parcela), 2)     AS total_pago,
        ROUND(AVG(valor_parcela), 2)     AS media_parcela
    FROM silver
    GROUP BY municipio, uf
    ORDER BY total_pago DESC
    LIMIT 20
""").df()

fig, ax = plt.subplots(figsize=(13, 6))
labels = df_p10['municipio'] + ' (' + df_p10['uf'] + ')'
ax.barh(labels[::-1], df_p10['total_pago'][::-1] / 1e6, color='steelblue')
ax.set_title('Top 20 municípios por total pago (R$ milhões)')
ax.set_xlabel('R$ milhões')
plt.tight_layout()
plt.savefig(f'{GOLD}/p10_top20_municipios_total.png', dpi=150)
plt.show()
df_p10

São Paulo lidera com R$ 889 milhões pagos no ano — quase o dobro do Rio de Janeiro. Mas a parcela média de SP é R$ 161, bem abaixo da nacional. É o oposto do que acontece em Porto Alegre (R$ 209) ou Campos dos Goytacazes (R$ 214): menos beneficiários, mas cada família recebendo mais. Volume alto com média baixa geralmente indica muitas famílias com pobreza menos severa.

### P11. Top 20 municípios por valor médio de parcela (mín. 1.000 pagamentos)

In [ ]:
df_p11 = con.execute("""
    SELECT
        municipio,
        uf,
        COUNT(*)                     AS pagamentos,
        ROUND(AVG(valor_parcela), 2) AS media_parcela
    FROM silver
    GROUP BY municipio, uf
    HAVING pagamentos >= 1000
    ORDER BY media_parcela DESC
    LIMIT 20
""").df()

fig, ax = plt.subplots(figsize=(13, 6))
labels = df_p11['municipio'] + ' (' + df_p11['uf'] + ')'
ax.barh(labels[::-1], df_p11['media_parcela'][::-1], color='darkorange')
ax.set_title('Top 20 municípios por parcela média (R$) — mín. 1.000 pagamentos')
ax.set_xlabel('R$')
plt.tight_layout()
plt.savefig(f'{GOLD}/p11_top20_municipios_media.png', dpi=150)
plt.show()
df_p11

Uiramutã, em Roraima, tem parcela média de R$ 445 — 2,3 vezes a média nacional. Todos os 20 primeiros municípios ficam no Acre, Amazonas, Pará ou Roraima, e a maioria são comunidades indígenas ou ribeirinhas. Uiramutã fica dentro da Terra Yanomami. O programa não ignora a intensidade da necessidade: paga mais onde as famílias têm mais filhos e menor renda.

### P12. Qual o total pago por região em cada mês? (evolução empilhada)

In [ ]:
df_p12 = con.execute("""
    SELECT
        mes_competencia                    AS mes,
        regiao,
        ROUND(SUM(valor_parcela) / 1e6, 2) AS total_pago_mi
    FROM silver
    GROUP BY mes, regiao
    ORDER BY mes, regiao
""").df()

pivot = df_p12.pivot(index='mes', columns='regiao', values='total_pago_mi')
pivot.plot(kind='bar', stacked=True, figsize=(14, 6), colormap='tab10')
plt.title('Total pago por região por mês (R$ milhões)')
plt.ylabel('R$ milhões')
plt.xticks(rotation=45)
plt.legend(loc='upper left')
plt.tight_layout()
plt.savefig(f'{GOLD}/p12_total_por_regiao_mes.png', dpi=150)
plt.show()

A composição regional não muda ao longo do ano — o Nordeste segura cerca de 51% do total em todos os meses. O salto de abril aparece em todas as regiões ao mesmo tempo e na mesma proporção. Não houve favorecimento regional na expansão: foi uma resposta uniforme, de norte a sul.

### P13. Quantos beneficiários receberam em todos os 12 meses? (beneficiários contínuos)

In [ ]:
df_p13 = con.execute("""
    WITH meses_por_nis AS (
        SELECT nis, COUNT(DISTINCT mes_competencia) AS meses_recebidos
        FROM silver
        GROUP BY nis
    )
    SELECT
        meses_recebidos,
        COUNT(*) AS beneficiarios,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM meses_por_nis
    GROUP BY meses_recebidos
    ORDER BY meses_recebidos
""").df()

fig, ax = plt.subplots()
ax.bar(df_p13['meses_recebidos'], df_p13['beneficiarios'] / 1e6, color='steelblue')
ax.set_title('Distribuição de meses recebidos por beneficiário')
ax.set_xlabel('Número de meses recebidos')
ax.set_ylabel('Beneficiários (milhões)')
ax.set_xticks(range(1, 13))
plt.tight_layout()
plt.savefig(f'{GOLD}/p13_continuidade_beneficiarios.png', dpi=150)
plt.show()
df_p13

86,8% dos beneficiários receberam em todos os 12 meses do ano. São 12,77 milhões de famílias para quem o Bolsa Família não foi um auxílio pontual — foi renda garantida durante o pior ano da pandemia. O pico em 9 meses (8,28%) provavelmente representa quem entrou no programa em abril e ficou até dezembro.

### P14. Qual a média de meses recebidos por beneficiário?

In [ ]:
con.execute("""
    WITH meses_por_nis AS (
        SELECT nis, COUNT(DISTINCT mes_competencia) AS meses_recebidos
        FROM silver
        GROUP BY nis
    )
    SELECT
        ROUND(AVG(meses_recebidos), 2)    AS media_meses,
        ROUND(MEDIAN(meses_recebidos), 0) AS mediana_meses,
        COUNT(*) AS total_beneficiarios
    FROM meses_por_nis
""").df()

Média de 11,42 meses, mediana de 12. Quando a mediana está no extremo superior da escala, a distribuição está muito concentrada nesse valor — o que bate com o que P13 mostrou: a esmagadora maioria recebeu o ano inteiro, e o restante puxa a média levemente para baixo.

### P15. Quais estados concentram maior percentual do total pago no país?

In [ ]:
df_p15 = con.execute("""
    WITH total AS (
        SELECT SUM(valor_parcela) AS grand_total FROM silver
    )
    SELECT
        uf,
        regiao,
        ROUND(SUM(valor_parcela) / 1e9, 2)                            AS total_pago_bilhoes,
        ROUND(100.0 * SUM(valor_parcela) / MAX(total.grand_total), 2) AS pct_nacional
    FROM silver, total
    GROUP BY uf, regiao
    ORDER BY pct_nacional DESC
""").df()

cor_map = {'Nordeste': '#e76f51', 'Sudeste': '#2a9d8f',
           'Norte': '#264653', 'Sul': '#f4a261', 'Centro-Oeste': '#e9c46a'}
cores = df_p15['regiao'].map(cor_map).fillna('gray')

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(df_p15['uf'], df_p15['pct_nacional'], color=cores)
ax.set_title('% do total pago nacional por estado')
ax.set_ylabel('% do total')
handles = [plt.Rectangle((0,0),1,1, color=v, label=k) for k, v in cor_map.items()]
ax.legend(handles=handles, loc='upper right')
plt.tight_layout()
plt.savefig(f'{GOLD}/p15_concentracao_por_uf.png', dpi=150)
plt.show()
df_p15

Os top 5 estados por gasto concentram 46,4% do total — bem menos do que eu esperaria antes de ver os dados. O programa é mais espalhado do que parece à primeira vista. O Nordeste como região soma 51%, mas isso é proporcional à concentração de pobreza estrutural da região, não uma distorção.

---
## Exportar Gold

In [ ]:
exports = {
    'pagamentos_por_mes.csv':       df_p3,
    'variacao_mensal.csv':          df_p4,
    'beneficiarios_por_uf.csv':     df_p5,
    'beneficiarios_por_regiao.csv': df_p6,
    'media_parcela_por_uf.csv':     df_p8,
    'top20_municipios_total.csv':   df_p10,
    'top20_municipios_media.csv':   df_p11,
    'continuidade.csv':             df_p13,
    'concentracao_por_uf.csv':      df_p15,
}

for nome, df in exports.items():
    df.to_csv(f'{GOLD}/{nome}', index=False)
    print(f'Exportado: {nome}')

print('\nGold completo.')